<a href="https://colab.research.google.com/github/Ena-AlexBrush/Fine-Tuning-Experiments/blob/main/GRPO_w_key_equal_prompt.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install datasets evaluate transformers[sentencepiece]
!pip install trl[GRPOTrainer]
!pip install --upgrade torchao

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 19.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 889.0/889.0 kB 46.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 20.1 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 52.7 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


In [3]:
from trl import GRPOTrainer, GRPOConfig
from datasets import load_dataset
import re

In [9]:

# 1. Load your dataset
train_dataset = load_dataset("hendzh/PromptShield", split="train[:50]")
eval_dataset = load_dataset("hendzh/PromptShield", split="validation[:50]")


In [16]:
# 2. Define a simple reward function
def format_reward_func(completions, **kwargs):
    """Reward function that checks if the completion has a specific format."""
    pattern = r"^<think>.*?</think><answer>.*?</answer>$"
    completion_contents = [completion for completion in completions]
    matches = [re.match(pattern, content, re.DOTALL) for content in completion_contents]
    return [1.0 if match else 0.0 for match in matches]

In [17]:

# 3. Configure training
training_args = GRPOConfig(
    output_dir="output",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=2,
    logging_steps=10,
    num_generations=4,
)

# 4. Initialize and train
trainer = GRPOTrainer(
    model="HuggingFaceTB/SmolLM2-135M",
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    reward_funcs=format_reward_func,
)

Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

In [ ]:
print(train_dataset[0].keys())

dict_keys(['prompt', 'quality', 'metadata', 'avg_rating', 'num_responses', 'agreement_ratio', 'raw_responses', 'kind', 'cluster_description', 'topic'])


In [18]:
trainer.train()

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 0}.


Step,Training Loss
10,0.000000
20,0.000000
30,0.000000
40,0.000000
50,0.000000
60,0.000000
70,0.000000
80,0.000000
90,0.000000
100,0.000000


Step,Training Loss
10,0.000000
20,0.000000
30,0.000000
40,0.000000
50,0.000000
60,0.000000
70,0.000000
80,0.000000
90,0.000000
100,0.000000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=150, training_loss=0.0, metrics={'train_runtime': 2107.9493, 'train_samples_per_second': 0.071, 'train_steps_per_second': 0.071, 'total_flos': 0.0, 'train_loss': 0.0, 'epoch': 3.0})

**Global_step**: The total number of updates steps performed during training, each each involves processing a batch of data and updating the model's weights

**training_loss**: final average loss value calculated across all the training steps.

**metrics**: dictionary contain various performance metrics:
*   **train_runtime**: the total time in secs that the training process took
*   **train_samples_per_second**: the average number of training samples processed per seconds.
*   **total_flos**: total floatin gpoint operations performed during training. Can be used to estimate computational cost.
*   **train_loss**: same as training_loss, representing the final average loss
*   **epoch**: teh number of full passes over the training dataset that were completed




In [19]:
trainer.evaluate()

Training Loss,Validation Loss,Step,Num Tokens,Completions/mean Length,Completions/min Length,Completions/max Length,Completions/clipped Ratio,Completions/mean Terminated Length,Completions/min Terminated Length,Completions/max Terminated Length,Rewards/format Reward Func/mean,Rewards/format Reward Func/std,Reward,Reward Std,Frac Reward Zero Std,Entropy,Clip Ratio/low Mean,Clip Ratio/high Mean,Clip Ratio/region Mean,Clip Ratio/low Min,Clip Ratio/high Max
0.000000,0.000000,150,222202.000000,197.860000,57.720000,256.000000,0.645000,89.723334,47.480000,141.880000,0.000000,0.000000,0.000000,0.000000,1.000000,3.465615,0.000000,0.000000,0.000000,0.000000,0.000000


{'eval_loss': 0.0,
 'eval_num_tokens': 222202.0,
 'eval_completions/mean_length': 197.86,
 'eval_completions/min_length': 57.72,
 'eval_completions/max_length': 256.0,
 'eval_completions/clipped_ratio': 0.645,
 'eval_completions/mean_terminated_length': 89.7233337020874,
 'eval_completions/min_terminated_length': 47.48,
 'eval_completions/max_terminated_length': 141.88,
 'eval_rewards/format_reward_func/mean': 0.0,
 'eval_rewards/format_reward_func/std': 0.0,
 'eval_reward': 0.0,
 'eval_reward_std': 0.0,
 'eval_frac_reward_zero_std': 1.0,
 'eval_entropy': 3.4656154346466064,
 'eval_clip_ratio/low_mean': 0.0,
 'eval_clip_ratio/high_mean': 0.0,
 'eval_clip_ratio/region_mean': 0.0,
 'eval_clip_ratio/low_min': 0.0,
 'eval_clip_ratio/high_max': 0.0}

**eval_loss**: the average loss calcuated on the evalution dtaaset. Similar to **training_loss**

**eval_num_tokens**: the total number of tokens generated or processed during evaluation.

**eval_completions/mean_length**: the average length of the generated completions
**eval_completions/max_length**: the max length among the generated completions

**eval_completions/min_length**: the min length among the generated completions
**eval_completions/clipped_ratio**: 69% of generated completions hit the 256 token cieling b4 naturally emitting and < eos > token

**eval_entropy**: measure how creative or diverse the model's token predictions are.

**Reward metrics**
**eval_reward**: Average reward, the mean score assigned by the reward function, higher the values indicate the model successfully matched the required format

**eval_reward_std**: Reward Variance, the variation in rewards across the generated completions.

**eval_frac_reward_zero_std**: zero variance fraction, 0 is ideal, GRPO requires distinct rewards withint a sample group to calculate relative advantages

**eval_completions/mean_terminated_length**: natural eos average. 31% of the responses that did complete naturally without hitting the cap and averaged around 87 tokens.
